# DQN on Atari Pong — Evaluation & Visualization

Loads the trained weights and produces:
1. A **GIF** of the greedy agent playing one episode
2. The **Average Reward** curve (Figure 2a, Nature 2015 style)
3. The **Average Max-Q on held-out states** curve (Figure 2b style)

In [ ]:
!pip install -q -U "gymnasium[atari]" "autorom[accept-rom-license]" ale-py imageio
!AutoROM --accept-license -q || true

In [ ]:
import sys, os, json
sys.path.append(os.getcwd())

import numpy as np, torch
import imageio.v2 as imageio
from IPython.display import Image as IPyImage, display

from src.env import make_env
from src.model import DQN
from src.config import DEVICE, SEED, SAVE_PATH

## 1. Load trained weights

In [ ]:
env = make_env()
policy = DQN(env.action_space.n).to(DEVICE)
policy.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
policy.eval()
env.close()
print("loaded weights from", SAVE_PATH)

## 2. Record a GIF of the greedy agent

In [ ]:
GIF_PATH = "assets/pong_agent.gif"
os.makedirs("assets", exist_ok=True)

def greedy_action(policy, state):
    with torch.no_grad():
        x = torch.as_tensor(np.asarray(state), device=DEVICE).unsqueeze(0)
        return policy(x).argmax(1).item()

def save_play_gif(policy, path=GIF_PATH, max_steps=5000, every=2, fps=30):
    env = make_env(render_mode="rgb_array")
    state, _ = env.reset(seed=SEED + 123)
    frames, total_reward = [], 0.0
    for t in range(max_steps):
        if t % every == 0:
            frame = env.render()
            if frame is not None:
                frames.append(frame)
        action = greedy_action(policy, state)
        state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        if terminated or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=fps)
    print(f"GIF saved to {path} | episode reward: {total_reward:+.0f} | frames: {len(frames)}")
    display(IPyImage(filename=path))
    return path, total_reward

save_play_gif(policy)

## 3. Plot training curves (Nature 2015 style)

In [ ]:
import matplotlib.pyplot as plt

with open("artifacts/history.json") as f:
    data = json.load(f)
history, returns = data["history"], data["returns"]

PLOT_REWARD_PATH = "assets/pong_average_reward.png"
PLOT_Q_PATH = "assets/pong_average_max_q.png"

# (a) Average reward per episode
plt.figure(figsize=(7, 4))
plt.plot(history["epoch"], history["reward100"])
plt.title("Average Reward on Pong")
plt.xlabel("Training epoch")
plt.ylabel("Average reward per episode (last 100)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOT_REWARD_PATH, dpi=150); plt.show()

# (b) Average max-Q on held-out states
plt.figure(figsize=(7, 4))
plt.plot(history["epoch"], history["q_mean"])
plt.title("Average Max-Q on Held-Out States")
plt.xlabel("Training epoch")
plt.ylabel("Average max Q")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(PLOT_Q_PATH, dpi=150); plt.show()

print("plots saved to", PLOT_REWARD_PATH, "and", PLOT_Q_PATH)